## Notebook Summary

This notebook documents a two-agent PC build assistant that uses a local CSV catalog to recommend either a full desktop build or a single hardware component, with one agent handling planning and validation and another handling live reference-rate conversion. In observed execution, the system asks follow-up questions when key details such as budget are missing, shows agent traces for its decisions, and returns warnings when compatibility checks are only partial. A key implementation challenge was working around incomplete compatibility metadata in the dataset while also depending on live exchange-rate calls and strict JSON parsing from LLM responses. Known limitations remain around socket inference, physical fit checks, PSU sizing, and the fact that converted prices are informational reference values rather than payment quotes. These constraints are surfaced directly so the notebook stays transparent and aligned with the Task 5 summary requirement.


The cells below mirror the top-level `agentic_system.py` message flow by importing the same public package interfaces, creating the same runtime objects, and running representative user requests through `run_turn(...)`. The stored outputs are from real observed runs and are included so a reviewer can inspect behavior, decision traces, limitations, and failure handling without rerunning every cell.


In [ ]:
import io
from contextlib import redirect_stdout

from pc_build_agent import CatalogTool, CurrencyAgent, DATA_DIR, LLMHelper, SessionState, run_turn
from pc_build_agent.conversation import print_logs

In [ ]:
catalog = CatalogTool(DATA_DIR)
llm_helper = LLMHelper()
currency_agent = CurrencyAgent()

In [ ]:
def run_example(message: str):
    state = SessionState()
    with redirect_stdout(io.StringIO()):
        response = run_turn(message, state, catalog, llm_helper, currency_agent)
    print("Assistant:")
    print(response)
    print()
    print_logs(state)
    return state

In [4]:
run_example('I need a custom PC for video editing.')

Assistant:
What budget should I use for the full build? If you have one exact number, I will treat it as a +/- 5% range.

Agent trace:
- User input received: I need a custom PC for video editing.
- Planner state after requirement extraction:
  {'intent': 'full_build',
   'use_case': 'video editing',
   'budget_target': None,
   'budget_min': None,
   'budget_max': None,
   'budget_currency': 'USD',
   'display_currency': 'USD',
   'requested_categories': [],
   'unsupported_categories': [],
   'preferences': {'include_peripherals': False}}
- Missing fields detected: budget


SessionState(intent='full_build', budget_target=None, budget_min=None, budget_max=None, budget_currency='USD', display_currency='USD', conversion_mode='usd_only', normalized_budget_target_usd=None, normalized_budget_min_usd=None, normalized_budget_max_usd=None, exchange_rate=None, exchange_rate_base=None, exchange_rate_date=None, exchange_rate_source=None, currency_warnings=[], use_case='video editing', preferences={'include_peripherals': False}, missing_fields=['budget'], current_build=None, revision_history=[], requested_categories=[], conversation_history=['I need a custom PC for video editing.'], logs=['User input received: I need a custom PC for video editing.', "Planner state after requirement extraction:\n{'intent': 'full_build',\n 'use_case': 'video editing',\n 'budget_target': None,\n 'budget_min': None,\n 'budget_max': None,\n 'budget_currency': 'USD',\n 'display_currency': 'USD',\n 'requested_categories': [],\n 'unsupported_categories': [],\n 'preferences': {'include_periphe

In [5]:
run_example('Recommend a printer for my setup.')

Assistant:
I cannot recommend printer because those items are not present in the local dataset. Please ask for a supported PC component such as a CPU, GPU, storage drive, monitor, or a full build.

Agent trace:
- User input received: Recommend a printer for my setup.
- Planner state after requirement extraction:
  {'intent': 'single_part',
   'use_case': None,
   'budget_target': None,
   'budget_min': None,
   'budget_max': None,
   'budget_currency': 'USD',
   'display_currency': 'USD',
   'requested_categories': [],
   'unsupported_categories': ['printer'],
   'preferences': {}}
- Unsupported categories requested: printer


SessionState(intent='single_part', budget_target=None, budget_min=None, budget_max=None, budget_currency='USD', display_currency='USD', conversion_mode='usd_only', normalized_budget_target_usd=None, normalized_budget_min_usd=None, normalized_budget_max_usd=None, exchange_rate=None, exchange_rate_base=None, exchange_rate_date=None, exchange_rate_source=None, currency_warnings=[], use_case=None, preferences={}, missing_fields=[], current_build=None, revision_history=[], requested_categories=[], conversation_history=['Recommend a printer for my setup.'], logs=['User input received: Recommend a printer for my setup.', "Planner state after requirement extraction:\n{'intent': 'single_part',\n 'use_case': None,\n 'budget_target': None,\n 'budget_min': None,\n 'budget_max': None,\n 'budget_currency': 'USD',\n 'display_currency': 'USD',\n 'requested_categories': [],\n 'unsupported_categories': ['printer'],\n 'preferences': {}}", 'Unsupported categories requested: printer'], unsupported_categori

In [6]:
run_example('Recommend a GPU under $400 for 1440p gaming.')

Assistant:
For a 1440p gaming video card within a $400 budget, I recommend the XFX Swift OC featuring the Radeon RX 9060 XT chipset. It offers 16GB memory, a base clock of 2220 MHz, and a boost clock up to 3320 MHz. The card is black and 290 mm in length, suitable for many mid-sized cases. No warnings or compatibility issues were found, and it fits within your specified budget. Please note, all prices are in USD, and the dataset assumes USD from an ambiguous "$" reference. Currency conversion uses a live reference exchange rate, but transaction rates may vary by bank or payment provider.

Budget/payment details:
- Original budget: $400.00  
- Video Card: XFX Swift OC ($399.99)  
- Total: $399.99  

Would you like to make any changes or consider other parts?

Agent trace:
- User input received: Recommend a GPU under $400 for 1440p gaming.
- Planner state after requirement extraction:
  {'intent': 'single_part',
   'use_case': '1440p gaming',
   'budget_target': 400.0,
   'budget_min': N

SessionState(intent='single_part', budget_target=400.0, budget_min=None, budget_max=400.0, budget_currency='USD', display_currency='USD', conversion_mode='usd_only', normalized_budget_target_usd=400.0, normalized_budget_min_usd=None, normalized_budget_max_usd=400.0, exchange_rate=1.0, exchange_rate_base='USD', exchange_rate_date=None, exchange_rate_source=None, currency_warnings=['Assuming USD from ambiguous "$" reference'], use_case='1440p gaming', preferences={'include_peripherals': False}, missing_fields=[], current_build=BuildProposal(selected_parts={'video-card': PartCandidate(category='video-card', name='XFX Swift OC', price=399.99, attributes={'price': 399.99, 'chipset': 'Radeon RX 9060 XT', 'memory': 16, 'core_clock': 2220, 'boost_clock': 3320, 'color': 'Black', 'length': 290})}, total_price=399.99, unmet_constraints=[], warnings=[], attempt=1), revision_history=[{'attempt': 1, 'build_total': 399.99, 'issues': [], 'warnings': []}], requested_categories=['video-card'], conversat

In [7]:
run_example('Build me a gaming PC around 120000 rupees with WiFi and 32 GB RAM.')

Assistant:
This gaming PC build includes a powerful Intel Core i7-13700F CPU alongside a Micro ATX ASRock B760M PG Riptide Wifi motherboard that supports WiFi as requested. The memory is 32 GB Silicon Power XPOWER Storm RGB, ideal for gaming and multitasking. Storage uses a spacious 1920 GB Patriot P310 SSD for fast load times. For graphics, the Gigabyte GAMING OC Radeon RX 9060 XT offers strong performance. The Thermaltake Toughpower GF1 (2024) 850W power supply ensures stable power delivery, and the Lian Li A3-mATX case adds style with tempered glass panels. Cooling is handled by the Geometric Future Eskimo Pro 42 CPU cooler, but note case clearance for the GPU and cooler height weren't verified due to data limitations. Also, storage lanes and PSU capacity are estimated, so keep those in mind.

Prices are in USD with INR as a live conversion reference:  
- CPU: $255.00 / INR 23,523.75  
- Motherboard: $119.99 / INR 11,069.08  
- Memory: $99.97 / INR 9,222.23  
- Internal Hard Drive: 

SessionState(intent='full_build', budget_target=120000.0, budget_min=114000.0, budget_max=126000.0, budget_currency='INR', display_currency='INR', conversion_mode='budget_and_display', normalized_budget_target_usd=1300.8, normalized_budget_min_usd=1235.76, normalized_budget_max_usd=1365.84, exchange_rate=92.25, exchange_rate_base='USD', exchange_rate_date='2026-03-16', exchange_rate_source='Frankfurter (ECB reference rates)', currency_warnings=[], use_case='gaming', preferences={'include_peripherals': False, 'memory_target_gb': 32, 'needs_wifi': True}, missing_fields=[], current_build=BuildProposal(selected_parts={'cpu': PartCandidate(category='cpu', name='Intel Core i7-13700F', price=255.0, attributes={'price': 255.0, 'core_count': 16, 'core_clock': 2.1, 'boost_clock': 5.2, 'microarchitecture': 'Raptor Lake', 'tdp': 65, 'graphics': None}), 'motherboard': PartCandidate(category='motherboard', name='ASRock B760M PG Riptide Wifi', price=119.99, attributes={'price': 119.99, 'socket': 'LGA